In [14]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH="telecom_guide.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n--- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

Loaded 9 pages from the PDF.

--- First page preview (first 500 chars) ---
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [16]:
print(pages[1].page_content[:500])

Telecom Technical Reference Guide  - Internal Use Only
1. Introduction to Mobile Networks
Mobile networks have evolved through several generations, each offering significant improvements in speed,
capacity, and capability.
2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to
around 50 kbps, sufficient only for text messaging and simple email.
3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, an


In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
  chunk_size = 600,
  chunk_overlap = 100,
  separators=["\n\n","\n"," ",""]
)

chunks = splitter.split_documents(pages)
len(chunks)

37

In [18]:
chunks[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [19]:
chunks[1].page_content

'Telecom Technical Reference Guide  - Internal Use Only\n1. Introduction to Mobile Networks\nMobile networks have evolved through several generations, each offering significant improvements in speed,\ncapacity, and capability.\n2G (GSM) networks introduced digital voice and basic data services such as SMS. Data speeds were limited to\naround 50 kbps, sufficient only for text messaging and simple email.\n3G (UMTS/HSPA) networks brought mobile broadband, enabling video calls, mobile internet browsing, and app'

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks,embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5315.12it/s]


Vector store ready. 74 vectors stored


In [21]:
# retrieval
retriever = vector_store.as_retriever(search_kwargs={"k":3})
test_query = "what is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved,1):
  print(f"--- chunk {i} ---")
  print(doc.page_content[:300])
  print()

--- chunk 1 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- chunk 2 ---
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 

--- chunk 3 ---
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt



#### rag pipeline

In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

#sytem prompt:
SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context doesn't contain enought information, say so clearly.

Context:{context}
"""

#prompt:
prompt = ChatPromptTemplate.from_messages([
  ("system",SYSTEM_PROMPT),
  ("human","{question}")
])


#llm:
llm = ChatGroq(
  model="openai/gpt-oss-120b",
  temperature=0,
  reasoning_format="parsed"
)

#helper function : format_docs
def format_docs(docs):
  return "\n\n---\n".join(doc.page_content for doc in docs)

#chain:
chain = (
  {"context":retriever | format_docs, "question" : RunnablePassthrough()} 
  | prompt
  | llm
  | StrOutputParser()
)

print("RAG CHAIN ASSEMBLED!")


RAG CHAIN ASSEMBLED!


In [23]:
question = "how does internation roaming work and what charges should I expect ?"
print(f"Q: {question}")
print(f"A: ",chain.invoke(question))

Q: how does internation roaming work and what charges should I expect ?
A:  International roaming works by having your device connect to a partner (visited) network when you leave the coverage area of your home network. The visited network authenticates you using an inter‑operator signalling protocol such as SS7 or Diameter. Your home network then validates your subscription and authorises the service, allowing you to use data, voice and SMS through the visited network.

**Charges you can expect**

* **Standard roaming rates** – You will be billed for any data, voice calls and SMS that you use while roaming, according to the rates set by your home operator for the visited country.  
* **Roaming bundles** – If you have purchased a roaming bundle, the bundle will cover usage for the period it is active.  
* **Potential overcharges** – Overcharges often occur when a roaming bundle was not active for the entire roaming period. The itemised bill will show the exact timestamps of when the bu